# 01 — Data Exploration

This notebook explores the EU AI Act PDF and the parsed graph data produced by the pipeline.

## Contents
1. Setup & Load Data
2. Raw PDF Text Inspection
3. Node Analysis
4. Edge Analysis
5. Graph Structure & Visualisation
6. Key Findings & Summary

---
## 1. Setup & Load Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import json
import yaml
from collections import Counter

# Style
sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
plt.rcParams["figure.dpi"] = 120
plt.rcParams["figure.figsize"] = (10, 5)

# Load config
with open("../configs/config.yaml", "r") as f:
    config = yaml.safe_load(f)

print("Config loaded ✓")

In [ ]:
# Load parsed text
with open(f"../{config['paths']['parsed_text']}", "r", encoding="utf-8") as f:
    parsed_pages = json.load(f)

# Load nodes and edges
nodes_df = pd.read_csv(f"../{config['paths']['nodes_csv']}")
edges_df = pd.read_csv(f"../{config['paths']['edges_csv']}")

print(f"Parsed pages:  {len(parsed_pages)}")
print(f"Nodes:         {len(nodes_df)}")
print(f"Edges:         {len(edges_df)}")

---
## 2. Raw PDF Text Inspection

Let's look at the raw extracted text to understand the document structure.

In [ ]:
# Basic document statistics
full_text = "\n\n".join(p["text"] for p in parsed_pages)
total_chars = len(full_text)
total_words = len(full_text.split())

page_lengths = [len(p["text"]) for p in parsed_pages]

print(f"Total characters:  {total_chars:,}")
print(f"Total words:       {total_words:,}")
print(f"Total pages:       {len(parsed_pages)}")
print(f"Avg chars/page:    {np.mean(page_lengths):,.0f}")
print(f"Min chars/page:    {np.min(page_lengths):,}")
print(f"Max chars/page:    {np.max(page_lengths):,}")

In [ ]:
# Page length distribution
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(1, len(page_lengths) + 1), page_lengths, color="steelblue", alpha=0.8)
ax.set_xlabel("Page Number")
ax.set_ylabel("Characters")
ax.set_title("EU AI Act — Text Length per Page")
ax.axhline(y=np.mean(page_lengths), color="red", linestyle="--", alpha=0.6, label=f"Mean ({np.mean(page_lengths):.0f})")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Show sample text from different sections of the document
print("=" * 80)
print("SAMPLE: Page 1 (Title / Preamble)")
print("=" * 80)
print(parsed_pages[0]["text"][:500])

print("\n" + "=" * 80)
print("SAMPLE: Page 51 (Articles section)")
print("=" * 80)
print(parsed_pages[50]["text"][:500])

---
## 3. Node Analysis

Examining the graph nodes extracted from the EU AI Act.

In [ ]:
# Node overview
print("Node columns:", list(nodes_df.columns))
print()
nodes_df.head(10)

In [ ]:
# Node type distribution
type_counts = nodes_df["type"].value_counts()
print("Node type distribution:")
print(type_counts)
print(f"\nTotal: {len(nodes_df)} nodes")

In [ ]:
# Node type bar chart
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette("viridis", len(type_counts))
type_counts.plot(kind="barh", ax=axes[0], color=colors)
axes[0].set_xlabel("Count")
axes[0].set_title("Node Count by Type")
for i, v in enumerate(type_counts.values):
    axes[0].text(v + 3, i, str(v), va="center", fontweight="bold")

# Pie chart
axes[1].pie(
    type_counts.values, labels=type_counts.index,
    autopct="%1.1f%%", colors=colors, startangle=90
)
axes[1].set_title("Node Type Proportions")

plt.tight_layout()
plt.show()

In [ ]:
# Text length analysis by node type
nodes_df["text_length"] = nodes_df["text"].str.len()
nodes_df["word_count"] = nodes_df["text"].str.split().str.len()

text_stats = nodes_df.groupby("type").agg(
    count=("text_length", "count"),
    mean_chars=("text_length", "mean"),
    median_chars=("text_length", "median"),
    max_chars=("text_length", "max"),
    mean_words=("word_count", "mean"),
).round(0)

print("Text length statistics by node type:")
text_stats

In [ ]:
# Text length distribution by node type (box plot)
fig, ax = plt.subplots(figsize=(12, 5))
order = nodes_df.groupby("type")["text_length"].median().sort_values(ascending=False).index
sns.boxplot(
    data=nodes_df, x="type", y="text_length", order=order,
    palette="viridis", showfliers=False, ax=ax
)
ax.set_xlabel("Node Type")
ax.set_ylabel("Text Length (characters)")
ax.set_title("Text Length Distribution by Node Type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Chapter distribution (for articles and paragraphs)
chapter_nodes = nodes_df[nodes_df["type"].isin(["article", "paragraph"])].copy()
chapter_counts = chapter_nodes["chapter"].value_counts().sort_index()

# Roman numeral sort
roman_order = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII", "XIII"]
chapter_counts = chapter_counts.reindex(roman_order).dropna().astype(int)

fig, ax = plt.subplots(figsize=(12, 5))
chapter_counts.plot(kind="bar", color=sns.color_palette("coolwarm", len(chapter_counts)), ax=ax)
ax.set_xlabel("Chapter")
ax.set_ylabel("Number of Nodes (Articles + Paragraphs)")
ax.set_title("Node Distribution across Chapters")
plt.xticks(rotation=0)
for i, v in enumerate(chapter_counts.values):
    ax.text(i, v + 1, str(v), ha="center", fontweight="bold", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Articles with the most paragraphs
para_per_article = (
    nodes_df[nodes_df["type"] == "paragraph"]
    .groupby("article_number")
    .size()
    .sort_values(ascending=False)
)

print("Top 15 articles by number of paragraphs:")
top15 = para_per_article.head(15)

fig, ax = plt.subplots(figsize=(12, 5))
top15.plot(kind="bar", color="steelblue", ax=ax)
ax.set_xlabel("Article Number")
ax.set_ylabel("Number of Paragraphs")
ax.set_title("Top 15 Articles by Paragraph Count")
plt.xticks(rotation=0)
for i, v in enumerate(top15.values):
    ax.text(i, v + 0.2, str(v), ha="center", fontweight="bold", fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# Definitions overview
definitions = nodes_df[nodes_df["type"] == "definition"].copy()
definitions["term"] = definitions["title"].str.replace("Definition: ", "", regex=False)

print(f"Total definitions: {len(definitions)}")
print(f"\nAll defined terms:")
for i, term in enumerate(definitions["term"].values, 1):
    print(f"  ({i:2d}) {term}")

---
## 4. Edge Analysis

Examining the relationships between nodes.

In [ ]:
# Edge overview
print("Edge columns:", list(edges_df.columns))
print()
edges_df.head(10)

In [ ]:
# Edge type distribution
relation_counts = edges_df["relation"].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = ["#2196F3", "#FF9800", "#4CAF50", "#9C27B0"]

# Bar chart
relation_counts.plot(kind="barh", color=colors, ax=axes[0])
axes[0].set_xlabel("Count")
axes[0].set_title("Edge Count by Relation Type")
for i, v in enumerate(relation_counts.values):
    axes[0].text(v + 10, i, str(v), va="center", fontweight="bold")

# Pie chart
axes[1].pie(
    relation_counts.values, labels=relation_counts.index,
    autopct="%1.1f%%", colors=colors, startangle=90
)
axes[1].set_title("Edge Type Proportions")

plt.tight_layout()
plt.show()

print(f"\nTotal edges: {len(edges_df)}")
print(f"Average edges per node: {len(edges_df) / len(nodes_df):.1f}")

In [ ]:
# Cross-reference analysis: which articles are most referenced?
refers_to_edges = edges_df[edges_df["relation"] == "refers_to"]

# Count incoming references per target
ref_target_counts = refers_to_edges["target"].value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 6))
ref_target_counts.plot(kind="barh", color="#FF9800", ax=ax)
ax.set_xlabel("Number of Incoming References")
ax.set_ylabel("Target Node")
ax.set_title("Top 20 Most Referenced Nodes (refers_to)")
ax.invert_yaxis()
for i, v in enumerate(ref_target_counts.values):
    ax.text(v + 0.3, i, str(v), va="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Definition usage: which terms are used most frequently?
uses_term_edges = edges_df[edges_df["relation"] == "uses_term"]

# Map definition IDs to term names
def_id_to_term = dict(zip(
    nodes_df[nodes_df["type"] == "definition"]["node_id"],
    nodes_df[nodes_df["type"] == "definition"]["title"].str.replace("Definition: ", "", regex=False)
))

term_usage = uses_term_edges["target"].map(def_id_to_term).value_counts().head(20)

fig, ax = plt.subplots(figsize=(14, 6))
term_usage.plot(kind="barh", color="#4CAF50", ax=ax)
ax.set_xlabel("Number of Nodes Using This Term")
ax.set_ylabel("Defined Term")
ax.set_title("Top 20 Most Frequently Used Defined Terms")
ax.invert_yaxis()
for i, v in enumerate(term_usage.values):
    ax.text(v + 0.3, i, str(v), va="center", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Cross-reference heatmap between chapters
# Map nodes to chapters
node_to_chapter = dict(zip(nodes_df["node_id"], nodes_df["chapter"]))

# For refers_to edges, get source/target chapters
ref_edges = refers_to_edges.copy()
ref_edges["source_chapter"] = ref_edges["source"].map(node_to_chapter)
ref_edges["target_chapter"] = ref_edges["target"].map(node_to_chapter)

# Drop NaN chapters (recitals, annexes)
ref_edges = ref_edges.dropna(subset=["source_chapter", "target_chapter"])

roman_order = ["I", "II", "III", "IV", "V", "VI", "VII", "VIII", "IX", "X", "XI", "XII", "XIII"]

# Build cross-reference matrix
cross_ref = pd.crosstab(
    ref_edges["source_chapter"], ref_edges["target_chapter"]
).reindex(index=roman_order, columns=roman_order, fill_value=0)

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cross_ref, annot=True, fmt="d", cmap="YlOrRd",
    linewidths=0.5, ax=ax, cbar_kws={"label": "Number of References"}
)
ax.set_xlabel("Target Chapter")
ax.set_ylabel("Source Chapter")
ax.set_title("Cross-Reference Heatmap between Chapters")
plt.tight_layout()
plt.show()

---
## 5. Graph Structure & Visualisation

Build a NetworkX graph to analyse structural properties.

In [ ]:
# Build NetworkX graph
G = nx.DiGraph()

# Add nodes with attributes
for _, row in nodes_df.iterrows():
    G.add_node(row["node_id"], type=row["type"], chapter=row.get("chapter"))

# Add edges
for _, row in edges_df.iterrows():
    if row["source"] in G and row["target"] in G:
        G.add_edge(row["source"], row["target"], relation=row["relation"])

print(f"Graph nodes: {G.number_of_nodes()}")
print(f"Graph edges: {G.number_of_edges()}")
print(f"Density:     {nx.density(G):.4f}")
print(f"Is connected (weakly): {nx.is_weakly_connected(G)}")

# Connected components
components = list(nx.weakly_connected_components(G))
print(f"Weakly connected components: {len(components)}")
print(f"Largest component size: {max(len(c) for c in components)}")
print(f"Isolated nodes: {sum(1 for c in components if len(c) == 1)}")

In [ ]:
# Degree distribution
in_degrees = dict(G.in_degree())
out_degrees = dict(G.out_degree())
total_degrees = {n: in_degrees[n] + out_degrees[n] for n in G.nodes()}

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# In-degree
axes[0].hist(in_degrees.values(), bins=30, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].set_xlabel("In-Degree")
axes[0].set_ylabel("Count")
axes[0].set_title("In-Degree Distribution")

# Out-degree
axes[1].hist(out_degrees.values(), bins=30, color="#FF9800", alpha=0.8, edgecolor="white")
axes[1].set_xlabel("Out-Degree")
axes[1].set_ylabel("Count")
axes[1].set_title("Out-Degree Distribution")

# Total degree
axes[2].hist(total_degrees.values(), bins=30, color="#4CAF50", alpha=0.8, edgecolor="white")
axes[2].set_xlabel("Total Degree")
axes[2].set_ylabel("Count")
axes[2].set_title("Total Degree Distribution")

plt.tight_layout()
plt.show()

print(f"Average in-degree:    {np.mean(list(in_degrees.values())):.2f}")
print(f"Average out-degree:   {np.mean(list(out_degrees.values())):.2f}")
print(f"Average total degree: {np.mean(list(total_degrees.values())):.2f}")
print(f"Max total degree:     {max(total_degrees.values())}")

In [ ]:
# Top nodes by degree (hub nodes)
degree_df = pd.DataFrame({
    "node_id": list(total_degrees.keys()),
    "total_degree": list(total_degrees.values()),
    "in_degree": [in_degrees[n] for n in total_degrees],
    "out_degree": [out_degrees[n] for n in total_degrees],
})
degree_df = degree_df.merge(nodes_df[["node_id", "type", "title"]], on="node_id", how="left")
degree_df = degree_df.sort_values("total_degree", ascending=False)

print("Top 20 nodes by total degree (most connected):")
degree_df.head(20)[["node_id", "type", "title", "in_degree", "out_degree", "total_degree"]]

In [ ]:
# Degree by node type
degree_by_type = degree_df.groupby("type").agg(
    count=("total_degree", "count"),
    mean_degree=("total_degree", "mean"),
    max_degree=("total_degree", "max"),
    mean_in=("in_degree", "mean"),
    mean_out=("out_degree", "mean"),
).round(2)

print("Degree statistics by node type:")
degree_by_type.sort_values("mean_degree", ascending=False)

In [ ]:
# Graph visualisation — article-level subgraph with cross-references
# (Full graph is too dense; show just articles + refers_to edges)

article_nodes = nodes_df[nodes_df["type"] == "article"]["node_id"].tolist()
article_refs = edges_df[
    (edges_df["relation"] == "refers_to") &
    (edges_df["source"].isin(article_nodes)) &
    (edges_df["target"].isin(article_nodes))
]

G_articles = nx.DiGraph()
for node_id in article_nodes:
    chapter = nodes_df[nodes_df["node_id"] == node_id]["chapter"].values[0]
    art_num = nodes_df[nodes_df["node_id"] == node_id]["article_number"].values[0]
    G_articles.add_node(node_id, chapter=chapter, art_num=int(art_num))

for _, row in article_refs.iterrows():
    G_articles.add_edge(row["source"], row["target"])

print(f"Article subgraph: {G_articles.number_of_nodes()} nodes, {G_articles.number_of_edges()} edges")

# Colour by chapter
chapter_colors = {
    "I": "#E53935", "II": "#D81B60", "III": "#8E24AA", "IV": "#5E35B1",
    "V": "#3949AB", "VI": "#1E88E5", "VII": "#039BE5", "VIII": "#00ACC1",
    "IX": "#00897B", "X": "#43A047", "XI": "#7CB342", "XII": "#FDD835",
    "XIII": "#FB8C00"
}

node_colors = [chapter_colors.get(G_articles.nodes[n].get("chapter", ""), "#999") for n in G_articles.nodes()]
node_sizes = [30 + G_articles.degree(n) * 15 for n in G_articles.nodes()]
labels = {n: str(G_articles.nodes[n]["art_num"]) for n in G_articles.nodes()}

fig, ax = plt.subplots(figsize=(16, 12))
pos = nx.spring_layout(G_articles, k=0.8, iterations=80, seed=42)

nx.draw_networkx_edges(G_articles, pos, alpha=0.15, arrows=True, arrowsize=8, ax=ax)
nx.draw_networkx_nodes(G_articles, pos, node_color=node_colors, node_size=node_sizes, alpha=0.85, ax=ax)
nx.draw_networkx_labels(G_articles, pos, labels, font_size=6, ax=ax)

# Legend
for ch, color in chapter_colors.items():
    ax.scatter([], [], c=color, s=60, label=f"Ch. {ch}")
ax.legend(loc="upper left", fontsize=7, ncol=2, title="Chapter")

ax.set_title("EU AI Act — Article Cross-Reference Network\n(nodes = articles, edges = refers_to)", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# PageRank analysis — which nodes are most "important" structurally?
pagerank = nx.pagerank(G, alpha=0.85)
pr_df = pd.DataFrame({
    "node_id": list(pagerank.keys()),
    "pagerank": list(pagerank.values())
}).merge(nodes_df[["node_id", "type", "title"]], on="node_id", how="left")

pr_df = pr_df.sort_values("pagerank", ascending=False)

print("Top 20 nodes by PageRank:")
pr_df.head(20)[["node_id", "type", "title", "pagerank"]]

In [ ]:
# PageRank by node type
fig, ax = plt.subplots(figsize=(10, 5))
pr_by_type = pr_df.groupby("type")["pagerank"].mean().sort_values(ascending=False)
pr_by_type.plot(kind="bar", color=sns.color_palette("viridis", len(pr_by_type)), ax=ax)
ax.set_ylabel("Mean PageRank")
ax.set_title("Average PageRank by Node Type")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## 6. Key Findings & Summary

In [ ]:
# Summary statistics
print("=" * 60)
print("  EU AI Act Graph — Summary Statistics")
print("=" * 60)
print()
print(f"  Document:")
print(f"    Pages:              {len(parsed_pages)}")
print(f"    Words:              {total_words:,}")
print(f"    Characters:         {total_chars:,}")
print()
print(f"  Graph:")
print(f"    Total nodes:        {G.number_of_nodes()}")
print(f"    Total edges:        {G.number_of_edges()}")
print(f"    Density:            {nx.density(G):.4f}")
print(f"    Avg degree:         {np.mean(list(total_degrees.values())):.1f}")
print(f"    Components:         {len(components)}")
print()
print(f"  Nodes by type:")
for ntype, count in type_counts.items():
    print(f"    {ntype:20s} {count:5d}")
print()
print(f"  Edges by relation:")
for rel, count in relation_counts.items():
    print(f"    {rel:20s} {count:5d}")
print()
print(f"  Top 5 hub nodes (by degree):")
for _, row in degree_df.head(5).iterrows():
    print(f"    {row['title'][:50]:50s}  degree={row['total_degree']}")
print()
print("=" * 60)

### Key Observations

1. **Document Scale**: The EU AI Act is a substantial document spanning 144 pages with 113 articles across 13 chapters, plus 13 annexes and 180+ recitals.

2. **Graph Density**: The graph captures ~974 nodes and ~3,873 edges, giving a rich relational structure for the GNN to learn from.

3. **Edge Composition**: Definition usage edges (`uses_term`) are the most common, followed by cross-references (`refers_to`). This suggests that term propagation is a key structural feature of the legislation.

4. **Hub Nodes**: Certain articles (especially core definitional and high-risk system articles) are heavily referenced, making them natural hubs in the graph.

5. **Chapter III Dominance**: Chapter III (High-Risk AI Systems) contains the most articles and paragraphs, reflecting it as the legislative core.

6. **GNN Implications**: The heterogeneous edge types and varying node granularity (article → paragraph → definition) provide multi-scale structural signals for the GNN to exploit during retrieval.